In [1]:
from datetime import datetime, timedelta
from utils.spark_session import get_sparkSession 
from utils.utilities import generate_date

In [2]:
from pyspark.sql.functions import current_timestamp, lit, expr, to_date, when, lower, upper, trim

In [3]:
spark = get_sparkSession("Load silver.fact_orders")

In [4]:
print("SPARK-APP : spark-UI - " + spark.sparkContext.uiWebUrl)

SPARK-APP : spark-UI - http://ba7b2bbb9aec:4040


In [5]:
from utils.load_controller import insert_control_logs, get_max_loadTimestamp

In [6]:
_schema_name = "silver"
_table_name = "fact_orders"
_fullname = f"{_schema_name}.{_table_name}"
_script_name = "silver_fact_orders.ipynb"
_bronze_table = "bronze.fact_orders"

print(f"SPARK-APP : Loading started for {_fullname}")

SPARK-APP : Loading started for silver.fact_orders


In [7]:
#spark config

spark.conf.set("spark.sql.shuffle.partitions",16)


In [8]:
#Read get_max_loadTimestamp

max_timestamp = get_max_loadTimestamp(spark, _schema_name, _table_name)


In [9]:
#Reading from bronze.raw_customer and de-duplicating the data

df = spark.read.table(_bronze_table) \
          .where(f"created_on > to_timestamp('{max_timestamp}')")

print(f"SPARK-APP: Bronze Table Data count : {df.count()}")

df_dedup = df.withColumn("_rn", expr("row_number() over (partition by order_id, line_num order by created_on desc)")) \
             .withColumnRenamed("product", "product_id") \
             .where("_rn = 1") \
             .drop("_rn")

loaded_count = df_dedup.count()

print(f"SPARK-APP: Table count after de-dupe : {loaded_count}")

SPARK-APP: Bronze Table Data count : 2
SPARK-APP: Table count after de-dupe : 2


In [10]:
df.show(10)
df.printSchema()


+--------+--------+----------+---------+-----------+------+-------+-------+--------+--------+----------+----------+---------------+-------------------------+----------+------------+--------------+--------------------+--------------------+--------------------+--------------------+---------------+
|order_id|line_num|order_date|ship_date|customer_id| store|channel|product|currency|quantity|unit_price|list_price|shipping_amount|shipping_paid_by_customer|tax_amount|order_status|payment_status|          created_on|          created_by|         modified_on|         modified_by|    source_file|
+--------+--------+----------+---------+-----------+------+-------+-------+--------+--------+----------+----------+---------------+-------------------------+----------+------------+--------------+--------------------+--------------------+--------------------+--------------------+---------------+
|   O1001|       1|  20250101| 20250102|    CUST001|AMZ_US|    MKT|  P1001|     USD|       2|       450|     

In [11]:
#Formating the data

df_ld = df_dedup.withColumn("line_num", expr("CAST(line_num as int)")) \
                .withColumn("order_date", to_date("order_date", 'yyyyMMdd')) \
                .withColumn("ship_date", to_date("ship_date", 'yyyyMMdd')) \
                .withColumn("quantity", expr("CAST(quantity as int)")) \
                .withColumn("unit_price", expr("CAST(unit_price as decimal(18,2))")) \
                .withColumn("list_price", expr("CAST(list_price as decimal(18,2))")) \
                .withColumn("shipping_amount", expr("CAST(shipping_amount as decimal(18,2))")) \
                .withColumn("shipping_paid_by_customer", when(lower("shipping_paid_by_customer") == "true", True).otherwise(False)) \
                .withColumn("tax_amount", expr("CAST(tax_amount as decimal(18,2))"))
                

In [12]:
#Standardizing the codes and types

df_ld = df_ld.withColumn("order_id", upper(trim("order_id"))) \
             .withColumn("customer_id", upper(trim("customer_id"))) \
             .withColumn("store", upper(trim("store"))) \
             .withColumn("channel", upper(trim("channel"))) \
             .withColumn("product_id", upper(trim("product_id"))) \
             .withColumn("currency", upper(trim("currency"))) \
             .withColumn("order_status", upper(trim("order_status"))) \
             .withColumn("payment_status", upper(trim("payment_status"))) 

In [13]:
#Adding audit columns

df_ld = df_ld.withColumn("created_on", current_timestamp()) \
             .withColumn("created_by", lit(_script_name)) \
             .withColumn("modified_on", current_timestamp()) \
             .withColumn("modified_by", lit(_script_name))

In [14]:
df_ld.show(10)
df_ld.printSchema()

+--------+--------+----------+----------+-----------+------+-------+----------+--------+--------+----------+----------+---------------+-------------------------+----------+------------+--------------+--------------------+--------------------+--------------------+--------------------+---------------+
|order_id|line_num|order_date| ship_date|customer_id| store|channel|product_id|currency|quantity|unit_price|list_price|shipping_amount|shipping_paid_by_customer|tax_amount|order_status|payment_status|          created_on|          created_by|         modified_on|         modified_by|    source_file|
+--------+--------+----------+----------+-----------+------+-------+----------+--------+--------+----------+----------+---------------+-------------------------+----------+------------+--------------+--------------------+--------------------+--------------------+--------------------+---------------+
|   O1001|       1|2025-01-01|2025-01-02|    CUST001|AMZ_US|    MKT|     P1001|     USD|       2|

In [15]:
#Writing to Table and log data to load_controller

_data = []

df_ld.write \
     .format("delta") \
     .mode("overwrite") \
     .saveAsTable(_fullname)
    
_data.append([_schema_name, _schema_name, _table_name, "full-load", "overwrite", str(datetime.now()), "success", loaded_count, _script_name, _script_name])

insert_control_logs(spark, _data)
    
print(f"SPARK-APP: Data written to {_schema_name}.{_table_name} and load_controller")

SPARK-APP: Data written to silver.fact_orders and load_controller


In [16]:
spark.read.table(f"{_schema_name}.{_table_name}").show()

+--------+--------+----------+----------+-----------+------+-------+----------+--------+--------+----------+----------+---------------+-------------------------+----------+------------+--------------+--------------------+--------------------+--------------------+--------------------+---------------+
|order_id|line_num|order_date| ship_date|customer_id| store|channel|product_id|currency|quantity|unit_price|list_price|shipping_amount|shipping_paid_by_customer|tax_amount|order_status|payment_status|          created_on|          created_by|         modified_on|         modified_by|    source_file|
+--------+--------+----------+----------+-----------+------+-------+----------+--------+--------+----------+----------+---------------+-------------------------+----------+------------+--------------+--------------------+--------------------+--------------------+--------------------+---------------+
|   O1001|       1|2025-01-01|2025-01-02|    CUST001|AMZ_US|    MKT|     P1001|     USD|       2|

In [17]:
spark.sql(f"select * from gold.load_controller where schema_name = '{_schema_name}' and table_name = '{_table_name}' order by created_on desc").show()

+------+-----------+-----------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
| layer|schema_name| table_name|load_type|write_mode|      load_timestamp|load_status|loaded_count|          created_on|          created_by|         modified_on|         modified_by|
+------+-----------+-----------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+
|silver|     silver|fact_orders|full-load| overwrite|2026-01-29 01:06:...|    success|           2|2026-01-29 01:06:...|silver_fact_order...|2026-01-29 01:06:...|silver_fact_order...|
+------+-----------+-----------+---------+----------+--------------------+-----------+------------+--------------------+--------------------+--------------------+--------------------+



In [18]:
from delta import DeltaTable

dt = DeltaTable.forName(spark,f"{_schema_name}.{_table_name}")

dt.history().limit(1).select("version","operationMetrics.numOutputRows").show(truncate=False)

+-------+-------------+
|version|numOutputRows|
+-------+-------------+
|0      |2            |
+-------+-------------+



In [19]:
spark.stop()